# 01: Exploratory data analysis (train set only)

Goal: understand the cleaned data and collect ideas for M4 feature engineering.
Use the **train** split only. The test set stays unseen until M6.

Run `make split` first if `load_split` says the file is missing.

In [1]:
%load_ext autoreload
%autoreload 2

import altair as alt
import altair_ally as aly
import pandas as pd

from credit_risk.data import BILL_AMTS, PAY_AMTS, PAY_STATUS, TARGET, load_split

# VegaFusion runs chart transforms (density, aggregates) in Python and embeds only
# the results, so the notebook doesn't store all ~24k training rows per chart
alt.data_transformers.enable("vegafusion")

CONTINUOUS = ["LIMIT_BAL", "AGE", *BILL_AMTS, *PAY_AMTS]
DISCRETE = ["SEX", "EDUCATION", "MARRIAGE", *PAY_STATUS]

train = load_split("train")
train.shape

(24000, 25)

## 1. Class balance

What share of clients default? Why does that make accuracy a poor metric here?

In [2]:
props = train[TARGET].value_counts(normalize=True).reset_index()

alt.Chart(props).mark_bar().encode(
    y=alt.Y("default:N", title="Default status", sort="-x"),
    x=alt.X("proportion:Q", title="Share of clients", axis=alt.Axis(format="%")),
    color=alt.Color("default:N", legend=None),
    tooltip=[
        alt.Tooltip("default:N", title="Default status"),
        alt.Tooltip("proportion:Q", title="Share", format=".1%"),
    ],
).properties(title="Target class imbalance in training data")

alt.Chart(...)

## 2. Continuous features by class

Do defaulters and non-defaulters differ in credit limit, age, bills or payments? Each class is normalized to 100%, so bar heights compare shapes, not counts. BILL_AMT and PAY_AMT use a signed log10 scale: 0 means zero, 3 about 1,000, 5 about 100,000, and negative values are credit balances.

In [3]:
def class_hist(data, col, log=False, maxbins=30, width=220, height=120):
    """Histogram of `col` with one overlaid bar series per class.

    Bar height is the share of that class's clients in each bin, so both classes
    sum to 100% and the 78/22 imbalance doesn't hide the default class.
    log=True bins sign(x) * log10(1 + |x|), which spreads out skewed amounts
    and keeps negative values (credit balances in BILL_AMT).
    """
    value = f"datum['{col}']"
    if log:
        value = f"({value} < 0 ? -1 : 1) * log(1 + abs({value})) / log(10)"
    return (
        alt.Chart(data[[col, TARGET]])
        .transform_calculate(
            value=value,
            status=f"datum['{TARGET}'] == 1 ? 'Default' : 'No default'",
        )
        .transform_bin(["bin_start", "bin_end"], "value", bin=alt.Bin(maxbins=maxbins))
        .transform_aggregate(n="count()", groupby=["status", "bin_start", "bin_end"])
        .transform_joinaggregate(class_n="sum(n)", groupby=["status"])
        .transform_calculate(share="datum.n / datum.class_n")
        .mark_bar(opacity=0.5)
        .encode(
            x=alt.X(
                "bin_start:Q",
                bin="binned",
                title=f"{col} (signed log10)" if log else col,
                axis=alt.Axis(format="~s"),
            ),
            x2="bin_end:Q",
            y=alt.Y("share:Q", title="Share of class", axis=alt.Axis(format="%"), stack=None),
            color=alt.Color("status:N", title="Default status", sort=["No default", "Default"]),
            tooltip=[
                alt.Tooltip("status:N", title="Status"),
                alt.Tooltip("bin_start:Q", title="From", format=",.2~f"),
                alt.Tooltip("bin_end:Q", title="To", format=",.2~f"),
                alt.Tooltip("share:Q", title="Share of class", format=".1%"),
                alt.Tooltip("n:Q", title="Clients", format=","),
            ],
        )
        .properties(width=width, height=height)
    )

In [4]:
class_hist(train, "LIMIT_BAL", maxbins=40, width=500, height=200).properties(
    title="Credit limit by default status"
)

alt.Chart(...)

In [5]:
alt.concat(
    class_hist(train, "LIMIT_BAL"),
    class_hist(train, "AGE"),
    *[class_hist(train, col, log=True) for col in [*BILL_AMTS, *PAY_AMTS]],
    columns=3,
).properties(title="Continuous features by default status (share of each class)")

alt.ConcatChart(...)

## 3. Categorical features

Counts per category mostly repeat the class imbalance, so the second chart shows the default rate per category instead. The dashed line is the overall rate; faded bars have fewer than 100 clients. PAY codes 3 to 8 are grouped as "3+" in the chart only.

In [6]:
# aly colours by a categorical, so cast the 0/1 target to string
eda = train.assign(**{TARGET: train[TARGET].astype(str)})

In [7]:
aly.dist(eda[[*DISCRETE, TARGET]].astype(str), dtype="object", color=TARGET)

alt.ConcatChart(...)

In [8]:
CATEGORY_ORDER = ["-2", "-1", "0", "1", "2", "3", "4", "3+"]

long = train.melt(id_vars=TARGET, value_vars=DISCRETE, var_name="variable", value_name="category")

# Group 3+ months late into one bucket for PAY_* only; codes 4-8 each have <100 clients
is_late = long["variable"].isin(PAY_STATUS) & (long["category"] >= 3)
long["category"] = long["category"].astype(str).where(~is_late, "3+")

rates = (
    long.groupby(["variable", "category"])[TARGET].agg(default_rate="mean", n="size").reset_index()
)
base_rate = train[TARGET].mean()

bars = (
    alt.Chart()
    .mark_bar()
    .encode(
        x=alt.X("category:N", title=None, sort=CATEGORY_ORDER),
        y=alt.Y("default_rate:Q", title="Default rate", axis=alt.Axis(format="%")),
        opacity=alt.condition("datum.n >= 100", alt.value(1), alt.value(0.4)),
        tooltip=[
            alt.Tooltip("category:N", title="Category"),
            alt.Tooltip("default_rate:Q", title="Default rate", format=".1%"),
            alt.Tooltip("n:Q", title="Clients", format=","),
        ],
    )
)
overall = alt.Chart().mark_rule(color="gray", strokeDash=[4, 4]).encode(y=alt.datum(base_rate))

alt.layer(bars, overall, data=rates).properties(width=180, height=140).facet(
    facet=alt.Facet("variable:N", title=None, sort=DISCRETE), columns=3
).resolve_scale(x="independent").properties(
    title=f"Default rate by category (dashed line = overall {base_rate:.1%})"
)

alt.FacetChart(...)

### 3a. The cleaned categories

EDUCATION (4 = other/unknown after cleaning) and MARRIAGE (3 = other).
Compare default rates per group. Does merging the undocumented codes look reasonable?

### 3b. Repayment status (PAY_1 to PAY_6)

-2 = no card use, -1 = paid in full, 0 = paid the minimum, 1 to 8 = months late.
How does the default rate change across these values? Is PAY_1 (most recent) more informative than PAY_6?

Note: code 1 (one month late) is common in PAY_1 but has 22 or fewer clients in PAY_2 to PAY_5 and none in PAY_6. The cause is undocumented; handle it in M4.

## 4. Bills and payments

Look at BILL_AMT and PAY_AMT against LIMIT_BAL. Try a quick version of credit utilization (BILL_AMT1 / LIMIT_BAL) and a payment ratio (PAY_AMT1 / BILL_AMT1). Watch for division by zero.

## 5. Correlations

Pearson and Spearman correlations (ID dropped). Expect BILL_AMT1 to BILL_AMT6 to be highly correlated with each other.

In [9]:
aly.corr(train.drop(columns="ID"))

alt.ConcatChart(...)

## 6. Feature ideas for M4

List 3 to 6 feature ideas in your own words, each with one sentence on why it might help predict default.